## Convert .mat file to netcdf for Mooring D Data 

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from scipy.io import loadmat
from datetime import datetime as dt
import glob
from tqdm import tqdm

In [ ]:
# Carson's Function to read MORSea matlab files
def read_MORSea_mat(filepath):
    '''
    Opens matlab file from MORSea that is assumed to contain 2 structs named 'dat' and 'hdr'.
    'hdr' must contain variables 'start_time','stop_time',and 'samp_int_minutes', from which the timebase will be constructed. If the resulting datetime array is longer than the data arrays, it is assumed that the start time is correct.
    An xarray dataset is created with all variables in 'dat' (assumes there are at least 2). Fields in 'hdr' are stored as metadata. 
    C. Witte 10/2025
    '''

    #open file and extract variable names
    mat = loadmat(filepath)
    data_fields = mat['dat'].dtype.names
    hdr_fields = mat['hdr'].dtype.names

    #construct timebase
    start = dt.strptime(mat['hdr']['start_time'][0][0][0].lstrip('0'), '%d/%m/%Y %H:%M:%S')
    stop = dt.strptime(mat['hdr']['stop_time'][0][0][0].lstrip('0'), '%d/%m/%Y %H:%M:%S') 
    interval = str(mat['hdr']['samp_int_minutes'][0][0][0][0])+'min'
    datetime_array = pd.date_range(start=start, end=stop, freq=interval)[:len(mat['dat'][data_fields[0]][0][0])] #force the datetime array to be the same length as the data
    diff = len(mat['dat'][data_fields[0]][0][0]) - len(datetime_array) #check if the datetime array is too short relative to the data and fix if needed
    if diff>0:
        datetime_array = pd.date_range(start=start, freq=interval, periods=len(datetime_array)+diff)
        
    #construct data_variables dictionary
    data_variables = {data_fields[0]:(('time','depth'),mat['dat'][data_fields[0]][0][0])}
    for idx in np.arange(1,len(data_fields)):
        name = data_fields[idx]
        data_variables[name] = (('time','depth'),mat['dat'][name][0][0])

    #construct attributes dictionary
    attrs = {hdr_fields[0]:mat['hdr'][hdr_fields[0]][0][0].item()}
    for idx in np.arange(1,len(hdr_fields)):
        name = hdr_fields[idx]
        attrs[name] = mat['hdr'][name][0][0].item()

    #construct xarray dataset
    out = xr.Dataset(data_vars=data_variables,
                     coords={'depth':[attrs['instr_depth_meters']],
                             'time':datetime_array},
                     attrs=attrs)
    
    return out

In [ ]:
folders = glob.glob('//thepenguin/penguin2/Data/TNB/mooring/Mooring_D_MORSea/Mooring_D_2014_2025/Mooring_D/*')
for folder in tqdm(folders):
    files = glob.glob(folder+'/*.mat')
    for file in files:
        ds = read_MORSea_mat(file)
        name = file.split('\\')[-1].split('.mat')[0] #extract filename to use for netcdf save
        ds.to_netcdf(folder+'/'+name+'.nc') 

Carson's function does not seem to automatically work for other Mooring D data folder, so I will try to follow his steps piecewise with one file to understand how everything works. I will be looking at the folder 'D_2017_2019'

In [ ]:
#open file and extract variable names
filepath = '//thepenguin/penguin2/Data/TNB/mooring/Mooring_D_MORSea/Mooring_D_2014_2025/Mooring_D/D_2017_2019/D_2017_RCM7_11199_576_m.mat'
mat = loadmat(filepath)
data_fields = mat['dat'].dtype.names
hdr_fields = mat['hdr'].dtype.names

In [ ]:
data_fields

In [ ]:
hdr_fields

In [ ]:
mat

In [ ]:
#construct timebase
start = dt.strptime(mat['hdr']['start_time'][0][0][0].lstrip('0'), '%d/%m/%Y %H:%M:%S')
stop = dt.strptime(mat['hdr']['stop_time'][0][0][0].lstrip('0'), '%d/%m/%Y %H:%M:%S') 
interval = str(mat['hdr']['samp_int_minutes'][0][0][0][0])+'min'
datetime_array = pd.date_range(start=start, end=stop, freq=interval)[:len(mat['dat'][data_fields[0]][0][0])] #force the datetime array to be the same length as the data
diff = len(mat['dat'][data_fields[0]][0][0]) - len(datetime_array) #check if the datetime array is too short relative to the data and fix if needed
if diff>0:
    datetime_array = pd.date_range(start=start, freq=interval, periods=len(datetime_array)+diff)

In [ ]:
start

In [ ]:
interval

In [ ]:
## Note that this is only for RCM instrument - the wrong interval was used above, should be 60 min, not 30 min. 
## Verified by looking at DateTime variable in original matlab file.
# interval = '60min'

In [ ]:
stop

In [ ]:
datetime_array = pd.date_range(start=start, end=stop, freq=interval)[:len(mat['dat'][data_fields[0]][0][0])] #force the datetime array to be the same length as the data
diff = len(mat['dat'][data_fields[0]][0][0]) - len(datetime_array) #check if the datetime array is too short relative to the data and fix if needed
if diff>0:
    datetime_array = pd.date_range(start=start, freq=interval, periods=len(datetime_array)+diff)

In [ ]:
datetime_array

In [ ]:
#construct data_variables dictionary
data_variables = {data_fields[0]:(('time','depth'),mat['dat'][data_fields[0]][0][0])}
for idx in np.arange(1,len(data_fields)):
    name = data_fields[idx]
    data_variables[name] = (('time','depth'),mat['dat'][name][0][0])

In [ ]:
# remove DateTime variables 
data_variables.pop('DateTime')

In [ ]:
data_variables

In [ ]:
#construct attributes dictionary
attrs = {hdr_fields[0]:mat['hdr'][hdr_fields[0]][0][0].item()}
for idx in np.arange(1,len(hdr_fields)):
    name = hdr_fields[idx]
    attrs[name] = mat['hdr'][name][0][0].item()

In [ ]:
attrs

In [ ]:
#construct xarray dataset
out = xr.Dataset(data_vars=data_variables,
                    coords={'depth':[attrs['instr_depth_meters']],
                            'time':datetime_array},
                    attrs=attrs)

In [ ]:
out

In [ ]:
D_2017_seaguard_1847_911_m = out 

In [ ]:
D_2017_seaguard_444_1135_m = out 

In [ ]:
D_2017_SBE39_30492_996_m = out 

In [ ]:
D_2017_SBE37_15451_772_m = out

In [ ]:
D_2017_SBE37_15450_1086_m = out

In [ ]:
D_2017_SBE16_1433_556_m = out 

In [ ]:
D_2017_RCM7_11199_576_m = out

In [ ]:
# D_2017_RCM7_11199_576_m.to_netcdf('//thepenguin/penguin2/Data/TNB/mooring/Mooring_D_MORSea/Mooring_D_2014_2025/Mooring_D/D_2017_2019/D_2017_RCM7_11199_576_m.nc')

In [ ]:
# save all datasets to netcdf 
names = ['D_2017_RCM7_11199_576_m',
         'D_2017_SBE37_15450_1086_m',
         'D_2017_SBE16_1433_556_m',
         'D_2017_SBE37_15451_772_m',
         'D_2017_SBE39_30492_996_m',
         'D_2017_seaguard_444_1135_m',
         'D_2017_seaguard_1847_911_m']
for name in names:
    ds = eval(name)
    ds.to_netcdf('//thepenguin/penguin2/Data/TNB/mooring/Mooring_D_MORSea/Mooring_D_2014_2025/Mooring_D/D_2017_2019/'+name+'.nc')

In [ ]:
D_2017_seaguard_444_1135_m